# 05 – EC3D Fine-tuning: Linear Probe vs Full Fine-tuning

**Obiettivo**: Valutare il pose encoder pre-addestrato (contrastive learning) su EC3D.

## Esperimenti:
1. **Linear Probe**: Encoder congelato + classificatore lineare
2. **MLP Probe**: Encoder congelato + MLP a 2 layer
3. **Fine-tuning**: Encoder trainabile + classificatore
4. **From Scratch**: Encoder inizializzato random + classificatore

## Baseline di riferimento (02_baselines.ipynb):
- Random Forest su feature handcrafted: **73.9% accuracy**

## Metriche:
- Accuracy Top-1 e Top-5
- Macro-F1
- Classification Report per classe


In [ ]:
import sys
from pathlib import Path
import pickle
import json
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    f1_score,
    top_k_accuracy_score,
    ConfusionMatrixDisplay
)

# Aggiungi il root al path per importare i modelli
ROOT_DIR = Path("..").resolve()
sys.path.insert(0, str(ROOT_DIR))

from models.pose_encoder.twostream_stgcn_plus import TwoStreamSTGCNPlusEncoder

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Seed per riproducibilità
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


## 1. Caricamento Dataset EC3D


In [ ]:
DATA_DIR = ROOT_DIR / "data" / "EC3D"
LOGS_DIR = ROOT_DIR / "logs"

# Carica sequenze
with open(DATA_DIR / "ec3d_sequences.pkl", "rb") as f:
    ec3d_data = pickle.load(f)

# Carica split cross-subject
with open(DATA_DIR / "split_cross_subject.json", "r") as f:
    split = json.load(f)

sequences = ec3d_data["sequences"]  # lista di np.array (T, 3, 25)
labels = ec3d_data["labels"]        # np.array (N_seq,)
meta = ec3d_data["meta"]            # lista di dict

train_indices = np.array(split["train_indices"])
test_indices = np.array(split["test_indices"])

print(f"Totale sequenze: {len(sequences)}")
print(f"Train: {len(train_indices)}, Test: {len(test_indices)}")
print(f"Numero classi: {len(np.unique(labels))}")
print(f"Esempio shape sequenza: {sequences[0].shape}")


In [ ]:
# Mapping ID -> Nome leggibile
ID_TO_NAME = {
    0: "SQUAT - Correct",
    1: "SQUAT - Feets too wide",
    2: "SQUAT - Knees inward",
    3: "SQUAT - Not low enough",
    4: "SQUAT - Front bended",
    5: "SQUAT - Unknown",
    6: "LUNGES - Correct",
    7: "LUNGES - Not low enough",
    8: "LUNGES - Knees pass toes",
    9: "PLANK - Correct",
    10: "PLANK - Banana back",
    11: "PLANK - Rolled back",
}

NUM_CLASSES = len(ID_TO_NAME)
print(f"Classi: {NUM_CLASSES}")

# Distribuzione classi nel train e test
train_labels = labels[train_indices]
test_labels = labels[test_indices]

print("\n📊 Distribuzione classi:")
print(f"{'Classe':<30} {'Train':>8} {'Test':>8}")
print("-" * 50)
for i in range(NUM_CLASSES):
    train_count = (train_labels == i).sum()
    test_count = (test_labels == i).sum()
    print(f"{ID_TO_NAME[i]:<30} {train_count:>8} {test_count:>8}")


## 2. Dataset e DataLoader PyTorch

Le sequenze hanno lunghezze variabili, quindi:
- Padding a lunghezza fissa (max_len)
- Trasposizione da (T, 3, 25) a (T, 25, 3) per il modello


In [ ]:
class EC3DDataset(Dataset):
    """
    Dataset PyTorch per EC3D.
    
    Trasforma le sequenze da (T, 3, 25) a (T, 25, 3) e applica padding.
    """
    def __init__(self, sequences, labels, indices, max_len=None):
        self.sequences = [sequences[i] for i in indices]
        self.labels = labels[indices]
        
        # Calcola max_len se non specificato
        if max_len is None:
            self.max_len = max(seq.shape[0] for seq in self.sequences)
        else:
            self.max_len = max_len
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        seq = self.sequences[idx]  # (T, 3, 25)
        label = self.labels[idx]
        
        # Trasposizione: (T, 3, 25) -> (T, 25, 3)
        seq = np.transpose(seq, (0, 2, 1))  # (T, 25, 3)
        
        T, V, C = seq.shape
        
        # Padding a max_len
        if T < self.max_len:
            pad = np.zeros((self.max_len - T, V, C), dtype=np.float32)
            seq = np.concatenate([seq, pad], axis=0)
        elif T > self.max_len:
            # Truncate (sampling uniforme)
            indices = np.linspace(0, T - 1, self.max_len).astype(int)
            seq = seq[indices]
        
        seq = torch.from_numpy(seq.astype(np.float32))
        label = torch.tensor(label, dtype=torch.long)
        
        return seq, label


In [ ]:
# Analisi lunghezze sequenze
seq_lengths = [seq.shape[0] for seq in sequences]
print(f"Lunghezza min: {min(seq_lengths)}, max: {max(seq_lengths)}, media: {np.mean(seq_lengths):.1f}")

# Usa una lunghezza fissa ragionevole
MAX_LEN = 150  # Copre la maggior parte delle sequenze
BATCH_SIZE = 16

# Crea dataset
train_dataset = EC3DDataset(sequences, labels, train_indices, max_len=MAX_LEN)
test_dataset = EC3DDataset(sequences, labels, test_indices, max_len=MAX_LEN)

# Crea DataLoader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"\n✅ Dataset creati:")
print(f"   Train: {len(train_dataset)} sequenze, {len(train_loader)} batch")
print(f"   Test: {len(test_dataset)} sequenze, {len(test_loader)} batch")

# Test
sample_seq, sample_label = train_dataset[0]
print(f"   Shape sequenza: {sample_seq.shape}")
print(f"   Label: {sample_label.item()} ({ID_TO_NAME[sample_label.item()]})")


## 3. Caricamento Pose Encoder Pre-addestrato


In [ ]:
# Carica il checkpoint migliore (epoch 30)
BEST_EPOCH = 30
CHECKPOINT_PATH = LOGS_DIR / f"pose_encoder_epoch{BEST_EPOCH}.pt"

print(f"Loading checkpoint: {CHECKPOINT_PATH}")
print(f"Exists: {CHECKPOINT_PATH.exists()}")

# Crea modello
def create_pose_encoder():
    return TwoStreamSTGCNPlusEncoder(
        input_dim=3,
        hidden_channels=[64, 128, 256, 256],
        output_dim=128,
        num_nodes=25,
        dropout=0.1,
        fusion_dropout=0.3
    )

# Carica pesi pre-addestrati
pretrained_encoder = create_pose_encoder()
state_dict = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
pretrained_encoder.load_state_dict(state_dict)
pretrained_encoder = pretrained_encoder.to(device)
pretrained_encoder.eval()

# Conta parametri
total_params = sum(p.numel() for p in pretrained_encoder.parameters())
print(f"\n✅ Pose encoder caricato: {total_params:,} parametri")

# Test forward pass
with torch.no_grad():
    test_input = torch.randn(2, MAX_LEN, 25, 3).to(device)
    test_output = pretrained_encoder(test_input)
    print(f"   Test input: {test_input.shape}")
    print(f"   Test output: {test_output.shape}")


## 4. Estrazione Embedding con Encoder Congelato


In [ ]:
@torch.no_grad()
def extract_embeddings(encoder, dataloader, device):
    """Estrae embedding per tutte le sequenze nel dataloader."""
    encoder.eval()
    embeddings = []
    all_labels = []
    
    for batch_seq, batch_labels in tqdm(dataloader, desc="Extracting embeddings"):
        batch_seq = batch_seq.to(device)
        emb = encoder(batch_seq, normalize=True)  # (B, 128)
        embeddings.append(emb.cpu().numpy())
        all_labels.append(batch_labels.numpy())
    
    embeddings = np.concatenate(embeddings, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    
    return embeddings, all_labels

# Estrai embedding
print("Estrazione embedding train...")
X_train_emb, y_train = extract_embeddings(pretrained_encoder, train_loader, device)

print("Estrazione embedding test...")
X_test_emb, y_test = extract_embeddings(pretrained_encoder, test_loader, device)

print(f"\n✅ Embedding estratti:")
print(f"   Train: {X_train_emb.shape}")
print(f"   Test: {X_test_emb.shape}")


## 5. Esperimento 1: Linear Probe (Encoder Congelato)

Addestra solo un classificatore lineare sulle embedding 128D.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Standardizza embedding
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_emb)
X_test_scaled = scaler.transform(X_test_emb)

# Linear Probe con Logistic Regression
linear_probe = LogisticRegression(
    max_iter=1000,
    multi_class='multinomial',
    solver='lbfgs',
    random_state=SEED,
    C=1.0
)

linear_probe.fit(X_train_scaled, y_train)

# Predizioni
y_pred_linear = linear_probe.predict(X_test_scaled)
y_proba_linear = linear_probe.predict_proba(X_test_scaled)

# Metriche
acc_linear = (y_pred_linear == y_test).mean()
macro_f1_linear = f1_score(y_test, y_pred_linear, average='macro')

# Top-5 accuracy
top5_acc_linear = top_k_accuracy_score(y_test, y_proba_linear, k=5, labels=list(range(NUM_CLASSES)))

print("=" * 50)
print("📊 LINEAR PROBE (Logistic Regression)")
print("=" * 50)
print(f"   Accuracy Top-1: {acc_linear:.3f}")
print(f"   Accuracy Top-5: {top5_acc_linear:.3f}")
print(f"   Macro-F1:       {macro_f1_linear:.3f}")


In [ ]:
print("\n📋 Classification Report (Linear Probe):")
print(classification_report(y_test, y_pred_linear, target_names=[ID_TO_NAME[i] for i in sorted(np.unique(y_test))]))


## 6. Esperimento 2: MLP Probe (Encoder Congelato)

Usa un MLP a 2 layer invece di un classificatore lineare.


In [ ]:
class MLPClassifier(nn.Module):
    """MLP classificatore a 2 layer."""
    def __init__(self, input_dim=128, hidden_dim=256, num_classes=12, dropout=0.3):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )
    
    def forward(self, x):
        return self.mlp(x)


def train_mlp_probe(X_train, y_train, X_test, y_test, 
                    hidden_dim=256, epochs=100, lr=1e-3, device='cpu'):
    """Addestra un MLP sulle embedding pre-estratte."""
    
    # Converti in tensori
    X_train_t = torch.from_numpy(X_train).float().to(device)
    y_train_t = torch.from_numpy(y_train).long().to(device)
    X_test_t = torch.from_numpy(X_test).float().to(device)
    y_test_t = torch.from_numpy(y_test).long().to(device)
    
    # Crea modello
    mlp = MLPClassifier(input_dim=X_train.shape[1], hidden_dim=hidden_dim, num_classes=NUM_CLASSES).to(device)
    
    optimizer = torch.optim.AdamW(mlp.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    best_acc = 0
    best_state = None
    history = {'train_loss': [], 'test_acc': []}
    
    for epoch in range(epochs):
        # Training
        mlp.train()
        optimizer.zero_grad()
        logits = mlp(X_train_t)
        loss = criterion(logits, y_train_t)
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        # Evaluation
        mlp.eval()
        with torch.no_grad():
            test_logits = mlp(X_test_t)
            test_preds = test_logits.argmax(dim=1)
            test_acc = (test_preds == y_test_t).float().mean().item()
        
        history['train_loss'].append(loss.item())
        history['test_acc'].append(test_acc)
        
        if test_acc > best_acc:
            best_acc = test_acc
            best_state = copy.deepcopy(mlp.state_dict())
        
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f} | Test Acc: {test_acc:.3f}")
    
    # Carica best model
    mlp.load_state_dict(best_state)
    
    return mlp, history, best_acc


In [ ]:
print("🚀 Training MLP Probe...\n")

mlp_probe, mlp_history, best_acc_mlp = train_mlp_probe(
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    hidden_dim=256,
    epochs=150,
    lr=1e-3,
    device=device
)

# Predizioni finali
mlp_probe.eval()
with torch.no_grad():
    X_test_t = torch.from_numpy(X_test_scaled).float().to(device)
    logits = mlp_probe(X_test_t)
    proba_mlp = F.softmax(logits, dim=1).cpu().numpy()
    y_pred_mlp = logits.argmax(dim=1).cpu().numpy()

# Metriche
acc_mlp = (y_pred_mlp == y_test).mean()
macro_f1_mlp = f1_score(y_test, y_pred_mlp, average='macro')
top5_acc_mlp = top_k_accuracy_score(y_test, proba_mlp, k=5, labels=list(range(NUM_CLASSES)))

print("\n" + "=" * 50)
print("📊 MLP PROBE (2-layer MLP)")
print("=" * 50)
print(f"   Accuracy Top-1: {acc_mlp:.3f}")
print(f"   Accuracy Top-5: {top5_acc_mlp:.3f}")
print(f"   Macro-F1:       {macro_f1_mlp:.3f}")


## 7. Esperimento 3: Fine-tuning Completo

Addestra encoder + classificatore end-to-end, partendo dai pesi pre-addestrati.


In [ ]:
class PoseClassifier(nn.Module):
    """Pose Encoder + Classificatore per fine-tuning."""
    def __init__(self, encoder, num_classes=12, freeze_encoder=False):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        
        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False
    
    def forward(self, x):
        emb = self.encoder(x, normalize=True)
        logits = self.classifier(emb)
        return logits


def train_classifier(model, train_loader, test_loader, 
                     epochs=50, lr=1e-4, device='cpu', desc="Training"):
    """Training loop per classificatore."""
    
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), 
                                   lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    best_acc = 0
    best_state = None
    history = {'train_loss': [], 'test_acc': [], 'test_f1': []}
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for batch_seq, batch_labels in train_loader:
            batch_seq = batch_seq.to(device)
            batch_labels = batch_labels.to(device)
            
            optimizer.zero_grad()
            logits = model(batch_seq)
            loss = criterion(logits, batch_labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        scheduler.step()
        
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch_seq, batch_labels in test_loader:
                batch_seq = batch_seq.to(device)
                logits = model(batch_seq)
                preds = logits.argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(batch_labels.numpy())
        
        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        test_acc = (all_preds == all_labels).mean()
        test_f1 = f1_score(all_labels, all_preds, average='macro')
        
        history['train_loss'].append(train_loss)
        history['test_acc'].append(test_acc)
        history['test_f1'].append(test_f1)
        
        if test_acc > best_acc:
            best_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
        
        if (epoch + 1) % 10 == 0:
            print(f"[{desc}] Epoch {epoch+1:3d} | Loss: {train_loss:.4f} | Acc: {test_acc:.3f} | F1: {test_f1:.3f}")
    
    model.load_state_dict(best_state)
    return model, history, best_acc


In [ ]:
print("🚀 Fine-tuning (encoder + classifier trainable)...\n")

# Crea modello con encoder pre-addestrato
encoder_ft = create_pose_encoder()
encoder_ft.load_state_dict(torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True))
model_ft = PoseClassifier(encoder_ft, num_classes=NUM_CLASSES, freeze_encoder=False).to(device)

trainable_params = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
print(f"Parametri trainabili: {trainable_params:,}\n")

model_ft, history_ft, best_acc_ft = train_classifier(
    model_ft, train_loader, test_loader,
    epochs=60, lr=5e-5, device=device, desc="Fine-tune"
)


In [ ]:
# Valutazione finale fine-tuning
model_ft.eval()
all_preds_ft, all_proba_ft, all_labels_ft = [], [], []

with torch.no_grad():
    for batch_seq, batch_labels in test_loader:
        batch_seq = batch_seq.to(device)
        logits = model_ft(batch_seq)
        proba = F.softmax(logits, dim=1).cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds_ft.extend(preds)
        all_proba_ft.extend(proba)
        all_labels_ft.extend(batch_labels.numpy())

y_pred_ft = np.array(all_preds_ft)
y_proba_ft = np.array(all_proba_ft)
y_test_ft = np.array(all_labels_ft)

acc_ft = (y_pred_ft == y_test_ft).mean()
macro_f1_ft = f1_score(y_test_ft, y_pred_ft, average='macro')
top5_acc_ft = top_k_accuracy_score(y_test_ft, y_proba_ft, k=5, labels=list(range(NUM_CLASSES)))

print("\n" + "=" * 50)
print("📊 FINE-TUNING (Pre-trained Encoder + Classifier)")
print("=" * 50)
print(f"   Accuracy Top-1: {acc_ft:.3f}")
print(f"   Accuracy Top-5: {top5_acc_ft:.3f}")
print(f"   Macro-F1:       {macro_f1_ft:.3f}")


## 8. Esperimento 4: Training From Scratch (Baseline)

Encoder inizializzato random per confronto con il pre-training.


In [ ]:
print("🚀 Training from scratch (random init)...\n")

encoder_scratch = create_pose_encoder()  # Pesi random
model_scratch = PoseClassifier(encoder_scratch, num_classes=NUM_CLASSES, freeze_encoder=False).to(device)

model_scratch, history_scratch, best_acc_scratch = train_classifier(
    model_scratch, train_loader, test_loader,
    epochs=100, lr=1e-4, device=device, desc="Scratch"
)


In [ ]:
# Valutazione finale from scratch
model_scratch.eval()
all_preds_scratch, all_proba_scratch = [], []

with torch.no_grad():
    for batch_seq, batch_labels in test_loader:
        batch_seq = batch_seq.to(device)
        logits = model_scratch(batch_seq)
        proba = F.softmax(logits, dim=1).cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds_scratch.extend(preds)
        all_proba_scratch.extend(proba)

y_pred_scratch = np.array(all_preds_scratch)
y_proba_scratch = np.array(all_proba_scratch)

acc_scratch = (y_pred_scratch == y_test).mean()
macro_f1_scratch = f1_score(y_test, y_pred_scratch, average='macro')
top5_acc_scratch = top_k_accuracy_score(y_test, y_proba_scratch, k=5, labels=list(range(NUM_CLASSES)))

print("\n" + "=" * 50)
print("📊 FROM SCRATCH (Random Init Encoder + Classifier)")
print("=" * 50)
print(f"   Accuracy Top-1: {acc_scratch:.3f}")
print(f"   Accuracy Top-5: {top5_acc_scratch:.3f}")
print(f"   Macro-F1:       {macro_f1_scratch:.3f}")


## 9. Riepilogo Risultati


In [ ]:
# Tabella riassuntiva
results = pd.DataFrame({
    'Method': [
        'Random Forest (handcrafted)',
        'Linear Probe (frozen encoder)',
        'MLP Probe (frozen encoder)',
        'Fine-tuning (pre-trained)',
        'From Scratch (random init)'
    ],
    'Top-1 Acc': [
        0.739,  # Baseline da 02_baselines.ipynb
        acc_linear,
        acc_mlp,
        acc_ft,
        acc_scratch
    ],
    'Top-5 Acc': [
        '-',
        f"{top5_acc_linear:.3f}",
        f"{top5_acc_mlp:.3f}",
        f"{top5_acc_ft:.3f}",
        f"{top5_acc_scratch:.3f}"
    ],
    'Macro-F1': [
        0.679,  # Baseline da 02_baselines.ipynb
        macro_f1_linear,
        macro_f1_mlp,
        macro_f1_ft,
        macro_f1_scratch
    ]
})

print("\n" + "=" * 70)
print("📊 RIEPILOGO RISULTATI - EC3D Classification (Cross-Subject)")
print("=" * 70)
print(results.to_string(index=False))
print("=" * 70)


In [ ]:
# Visualizzazione risultati
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

methods = ['RF\n(baseline)', 'Linear\nProbe', 'MLP\nProbe', 'Fine-\ntuning', 'From\nScratch']
top1_accs = [0.739, acc_linear, acc_mlp, acc_ft, acc_scratch]
macro_f1s = [0.679, macro_f1_linear, macro_f1_mlp, macro_f1_ft, macro_f1_scratch]
colors = ['#95a5a6', '#3498db', '#2980b9', '#27ae60', '#e74c3c']

# Top-1 Accuracy
ax1 = axes[0]
bars1 = ax1.bar(methods, top1_accs, color=colors, edgecolor='black', linewidth=1.2)
ax1.set_ylabel('Top-1 Accuracy', fontsize=12)
ax1.set_title('Classification Accuracy on EC3D', fontsize=14, fontweight='bold')
ax1.set_ylim(0, 1)
ax1.axhline(y=0.739, color='gray', linestyle='--', alpha=0.7, label='RF Baseline')
for bar, val in zip(bars1, top1_accs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Macro-F1
ax2 = axes[1]
bars2 = ax2.bar(methods, macro_f1s, color=colors, edgecolor='black', linewidth=1.2)
ax2.set_ylabel('Macro-F1', fontsize=12)
ax2.set_title('Macro-F1 Score on EC3D', fontsize=14, fontweight='bold')
ax2.set_ylim(0, 1)
ax2.axhline(y=0.679, color='gray', linestyle='--', alpha=0.7, label='RF Baseline')
for bar, val in zip(bars2, macro_f1s):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(ROOT_DIR / 'debug_plots' / 'ec3d_finetuning_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Plot salvato in debug_plots/ec3d_finetuning_comparison.png")


In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax1 = axes[0]
ax1.plot(history_ft['train_loss'], label='Fine-tuning', color='#27ae60', linewidth=2)
ax1.plot(history_scratch['train_loss'], label='From Scratch', color='#e74c3c', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss')
ax1.set_title('Training Loss Comparison', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy curves
ax2 = axes[1]
ax2.plot(history_ft['test_acc'], label='Fine-tuning', color='#27ae60', linewidth=2)
ax2.plot(history_scratch['test_acc'], label='From Scratch', color='#e74c3c', linewidth=2)
ax2.axhline(y=acc_linear, color='#3498db', linestyle='--', label='Linear Probe', linewidth=2)
ax2.axhline(y=0.739, color='gray', linestyle=':', label='RF Baseline', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Test Accuracy')
ax2.set_title('Test Accuracy Comparison', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT_DIR / 'debug_plots' / 'ec3d_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Conclusioni

### Risultati Chiave:

1. **Linear Probe**: Testa quanto bene le embedding del pose encoder pre-addestrato discriminano le classi EC3D senza alcun fine-tuning.

2. **MLP Probe**: Aggiunge capacità non-lineare al classificatore mantenendo l'encoder congelato.

3. **Fine-tuning**: Adatta l'intero modello al dominio EC3D, partendo dai pesi pre-addestrati.

4. **From Scratch**: Baseline per quantificare il vantaggio del pre-training contrastivo.

### Interpretazione:

- Se **Fine-tuning >> From Scratch**: il pre-training ha trasferito conoscenza utile.
- Se **Linear Probe ≈ Fine-tuning**: le embedding sono già molto discriminative.
- Se **Fine-tuning >> Linear Probe**: serve adattamento al dominio specifico.

### Prossimi Passi:
- Analizzare gli errori per classe
- Provare data augmentation
- Valutare su altri split o dataset


In [ ]:
# Salva risultati in CSV
results_path = ROOT_DIR / 'debug_outputs' / 'ec3d_finetuning_results.csv'
results.to_csv(results_path, index=False)
print(f"✅ Risultati salvati in: {results_path}")

# Classification report fine-tuning
print("\n" + "=" * 70)
print("📋 CLASSIFICATION REPORT - Fine-tuning")
print("=" * 70)
print(classification_report(y_test_ft, y_pred_ft, 
                           target_names=[ID_TO_NAME[i] for i in sorted(np.unique(y_test_ft))]))
